# Node Breaker Models

This section describes transmission node-breaker modeling concepts and the CIMantic Graphs `NodeBreakerModel` class for accessing CIM transmission models.

All diagrams on this page are generated as [Mermaid](https://mermaid.js.org/) text with the `cimgraph.utils` module of CIMantic Graphs and displayed as a Markdown `mermaid` block:

In [1]:
from IPython.display import Markdown, display

from cimgraph import utils

def show_mermaid(diagram):
    display(Markdown(f'```mermaid\n{diagram}\n```'))

## Transmission Network Models in CIM

CIM was originally designed to support full node-breaker modeling of transmission networks and substations. In a node-breaker model, every breaker, disconnector and busbar section in a substation is represented, connected through `Terminal` and `ConnectivityNode` objects. Equipment is contained in a hierarchy of `GeographicalRegion` > `SubGeographicalRegion` > `Substation` > `VoltageLevel` (and optionally `Bay`).

----

## Loading a NodeBreakerModel from an XML File

CIMantic Graphs can be used without any database by building the graph directly from an XML file. This page uses the maple 10 bus node-breaker test model.

The first step is to import the CIM profile and set the matching environment variable used by CIM-Graph. For an explanation of CIM profiles, see [Profiles Overview](../02_cim_profiles/2_1_profiles_overview.ipynb).

The test file contains classes that are not in the `cim17v40` profile (e.g. `PhotovoltaicUnit`). Setting `CIMG_VALIDATION_LOG_LEVEL` to `DEBUG` logs the messages about them at debug level, so they are not printed:

In [2]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cim17v40'  # import and env var must match
os.environ['CIMG_VALIDATION_LOG_LEVEL'] = 'DEBUG'

import cimprofiles.cim17.cim17v40 as cim  # import and env var must match

Next, use the `XMLFile` interface to open the XML file. The namespaces declared in the file header are read automatically. If custom extensions are used, the `namespaces` argument adds prefix-to-URI mappings for them.

In [3]:
from cimgraph.databases import XMLFile

file = XMLFile(filename='../../tests/test_models/maple10nodebreaker.xml')

Finally, create a new `NodeBreakerModel` to load the transmission model into the graph. With `container=None` the whole file is loaded:

In [4]:
from cimgraph.models import NodeBreakerModel

network = NodeBreakerModel(container=None, connection=file)

The substation hierarchy can be read by following the associations between the containers:

In [5]:
for substation in network.list_by_class(cim.Substation):
    print(substation.name, 'in region', substation.Region.name)
    for voltage_level in substation.VoltageLevels:
        nominal_kv = voltage_level.BaseVoltage.nominalVoltage.to('kV')
        print(f'    {voltage_level.name}: {nominal_kv} kV')

maple10bus_sub1 in region small
    SUB1_230.0_B1: 230.0 kV
    SUB1_115.0_B1: 115.0 kV
    SUB1_34.5_B1: 34.5 kV
    SUB1_12.47_B1: 12.47 kV
maple10bus_sub2 in region small
    SUB2_230.0_B1: 230.0 kV
    SUB2_115.0_B1: 115.0 kV
    SUB2_34.5_B1: 34.5 kV
    SUB2_4.16_B1: 4.16 kV


A sample breaker object and its associations:

In [6]:
# Get first breaker in graph
breaker = network.first(cim.Breaker)
# Display the breaker
show_mermaid(utils.get_mermaid(breaker))

```mermaid
mindmap
    276809DF((**Breaker**
        name: hv_bk_1
        normalOpen: False
        open: False
        retained: True
        ratedCurrent: 400 ampere
        breakingCapacity: 400 ampere))
        [Location]
            6A85EE07((**Location**
                name: hv_bk_1_Loc))
        [EquipmentContainer]
            BB020444((**VoltageLevel**
                name: SUB1_230.0_B1))
        [BaseVoltage]
            91E72B6A((**BaseVoltage**
                name: BaseV_230.0000))
        ["Terminals"]
            CE6DD7CA((**Terminal**
                name: hv_bk_1_T1))
            88C03456((**Terminal**
                name: hv_bk_1_T2))

```

## Modifying Models

Objects can be created, modified, or deleted using the `.create(cim_class, **kwargs)`, `.modify(object, attribute, value)` and `.delete(object)` methods. Each change is recorded in `network.incrementals`, and the optional `incremental_file` argument writes every change recorded so far to a CIM difference file. See [Using Incrementals](../02_cim_profiles/2_4_using_incrementals.ipynb) for details.

In [7]:
new_breaker = network.create(cim.Breaker, name='new_breaker', open=False)
print(new_breaker.name, new_breaker.identifier in network.graph[cim.Breaker])

new_breaker True


In [8]:
network.delete(new_breaker)
print(new_breaker.identifier in network.graph[cim.Breaker])

False


A breaker that was created and then deleted leaves no difference. Opening the existing breaker does:

In [9]:
network.modify(breaker, 'open', True, incremental_file='breaker_incremental.xml')

with open('breaker_incremental.xml') as incremental:
    print(incremental.read())

<?xml version="1.0" encoding="utf-8"?>
<rdf:RDF xmlns:cim="http://iec.ch/TC57/CIM100#" xmlns:md="http://iec.ch/TC57/61970-552/ModelDescription/1#" xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:dm="http://iec.ch/2002/schema/CIM_difference_model#">
<dm:DifferenceModel rdf:about="urn:uuid:8afb6a14-aa2e-48db-a32a-8549cceaf53e">
  <dm:forwardDifferences rdf:parseType="Statements">
    <rdf:Description rdf:about="urn:uuid:276809DF-6493-4DAE-A7BB-AD78A5F0DA95">
      <cim:Switch.open>true</cim:Switch.open>
    </rdf:Description>
  </dm:forwardDifferences>
  <dm:reverseDifferences rdf:parseType="Statements">
    <rdf:Description rdf:about="urn:uuid:276809DF-6493-4DAE-A7BB-AD78A5F0DA95">
      <cim:Switch.open>false</cim:Switch.open>
    </rdf:Description>
  </dm:reverseDifferences>
</dm:DifferenceModel>
</rdf:RDF>



## Saving to a File

Use the `utils.write_xml()` and `utils.write_json_ld()` methods to save the modified network to a file:

In [10]:
utils.write_xml(network, 'maple10nodebreaker_modified.xml')
utils.write_json_ld(network, 'maple10nodebreaker_modified.jsonld')

-----

## Distributed Modeling

`NodeBreakerModel(..., distributed=True)` is not implemented yet: it is intended to build a separate typed property graph for each substation and voltage level, but currently builds no areas. Use the centralized model, or [distributed feeder areas](4_5_distributed_area.ipynb) for distribution feeders.